# Demand Retail — Entrenamiento e Inferencia con SageMaker BYOC

Este notebook ejecuta el flujo completo de Bring Your Own Container (BYOC) en Amazon SageMaker:
1. Build del container y push a ECR
2. Subida de datos de entrenamiento a S3
3. Training job con SageMaker Estimator
4. Deploy del endpoint de inferencia en tiempo real
5. Predicciones de prueba
6. Limpieza del endpoint

## 1. Configuración del entorno

In [1]:
import boto3
import os
import numpy as np
import pandas as pd
import sagemaker as sage
from pathlib import Path
from sagemaker import get_execution_role
from sagemaker.serializers import CSVSerializer
from time import gmtime, strftime

# IAM role con permisos de SageMaker + ECR + S3
role = get_execution_role()

# Sesión de SageMaker
sess = sage.Session()

# Bucket y prefix S3 donde se subirán datos y artefactos
default_bucket        = 'sagemaker-demand-retail'
default_bucket_prefix = sess.default_bucket_prefix
prefix                = "byoc-v1"

if default_bucket_prefix:
    prefix = f"{default_bucket_prefix}/{prefix}"

# Nombre del container — debe coincidir con el repositorio en ECR
IMAGE_NAME = "demand-retail-training"

account = sess.boto_session.client("sts").get_caller_identity()["Account"]
region  = sess.boto_session.region_name
image   = f"{account}.dkr.ecr.{region}.amazonaws.com/{IMAGE_NAME}:latest"

print(f"Role:    {role}")
print(f"Bucket:  {default_bucket}")
print(f"Prefix:  {prefix}")
print(f"Image:   {image}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Role:    arn:aws:iam::433956820129:role/SageMakerStudioExecutionRole2026
Bucket:  sagemaker-demand-retail
Prefix:  byoc-v1
Image:   433956820129.dkr.ecr.us-east-1.amazonaws.com/demand-retail-training:latest


## 2. Build del container y push a ECR

Ejecuta el script `build_and_push.sh` desde el directorio `training/` del repositorio.
El script construye la imagen Docker y la sube al repositorio ECR `demand-retail-training`.

In [2]:
# Ajusta esta ruta al directorio training/ de tu repositorio
TRAINING_DIR = os.getcwd()
print(f"Directorio de trabajo: {TRAINING_DIR}")

Directorio de trabajo: /home/sagemaker-user/Demand_Retail/src/training


In [3]:
%%sh
# Permisos de ejecución a los scripts del container
chmod +x src/train
chmod +x src/serve

In [4]:
%%sh
# Build y push a ECR

image="demand-retail-training"

account=$(aws sts get-caller-identity --query Account --output text)
region=$(aws configure get region)
region=${region:-us-east-1}
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# Crear repositorio en ECR si no existe
aws ecr describe-repositories --repository-names "${image}" > /dev/null 2>&1
if [ $? -ne 0 ]; then
    aws ecr create-repository --repository-name "${image}" > /dev/null
fi

# Login a ECR
aws ecr get-login-password --region "${region}" | \
    docker login --username AWS --password-stdin "${account}.dkr.ecr.${region}.amazonaws.com"

# Build desde el directorio training/ donde está el Dockerfile
docker build --network sagemaker -t ${image} .

# Tag y push
docker tag ${image} ${fullname}
docker push ${fullname}

echo "Image subida: ${fullname}"

WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores



Login Succeeded


DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.



Sending build context to Docker daemon  185.3kB
Step 1/9 : FROM python:3.11-slim
 ---> e67db9b14d09
Step 2/9 : RUN apt-get -y update && apt-get install -y --no-install-recommends         nginx         ca-certificates     && rm -rf /var/lib/apt/lists/*
 ---> Using cache
 ---> 7cb521ec91a3
Step 3/9 : RUN pip --no-cache-dir install         polars==1.37.1         scikit-learn==1.8.0         pyarrow==23.0.0         joblib==1.5.0         numpy==2.4.1         pandas==2.3.3         xgboost==3.1.3         category-encoders==2.9.0         flask         gunicorn
 ---> Using cache
 ---> 62ded1266065
Step 4/9 : ENV PYTHONUNBUFFERED=TRUE
 ---> Using cache
 ---> fe6506870a09
Step 5/9 : ENV PYTHONDONTWRITEBYTECODE=TRUE
 ---> Using cache
 ---> abc0407ee275
Step 6/9 : ENV PATH="/opt/program:${PATH}"
 ---> Using cache
 ---> a13302e5d4e3
Step 7/9 : COPY src/ /opt/program
 ---> 6c65981589b5
Step 8/9 : WORKDIR /opt/program
 ---> Running in 291f97860cfd
 ---> Removed intermediate container 291f97860cfd
 --->

## 3. Subida de datos de entrenamiento a S3

SageMaker descarga los datos del canal `training` al path `/opt/ml/input/data/training/` dentro del container.
Sube el archivo `monthly_sales.parquet` junto con los catálogos auxiliares que necesita `predictor.py` al momento de serving.

In [5]:
BASE_DIR = Path(os.getcwd()).parent.parent
print(f"Directorio de trabajo: {BASE_DIR}")

Directorio de trabajo: /home/sagemaker-user/Demand_Retail


In [7]:
# Directorio local con los datos de entrenamiento
# Debe contener:
#   monthly_sales.parquet           ← datos de entrenamiento
#   items_catalog.parquet           ← catálogo de productos (necesario en serving)
#   items_categories_catalog.parquet
#   shops_catalog.parquet
DATA_DIR = str(BASE_DIR / "data" / "prep")

data_location = sess.upload_data(
    DATA_DIR,
    bucket=default_bucket,
    key_prefix=f"{prefix}/input"
)

print(f"Datos subidos a: {data_location}")

Datos subidos a: s3://sagemaker-demand-retail/byoc-v1/input


## 4. Training job con SageMaker Estimator

SageMaker invoca el container con el comando `train`. El script `/opt/program/train` lee los datos
desde `/opt/ml/input/data/training/` y escribe el modelo en `/opt/ml/model/`.
Al terminar, SageMaker empaqueta `/opt/ml/model/` en un `.tar.gz` y lo sube a S3.

In [8]:
s3_output_path = f"s3://{default_bucket}/{prefix}/output"

estimator = sage.estimator.Estimator(
    image_uri=image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=s3_output_path,
    sagemaker_session=sess,
    # Hiperparámetros — train los lee desde hyperparameters.json (siempre strings)
    hyperparameters={
        "n_estimators":  "400",
        "learning_rate": "0.05",
        "max_depth":     "9",
        "random_state":  "42",
        "monthly_file":  "monthly_sales.parquet",
    }
)

estimator.fit({"training": data_location})
print("Training job completado")

INFO:sagemaker:Creating training-job with name: demand-retail-training-2026-04-03-22-21-00-644


2026-04-03 22:21:02 Starting - Starting the training job...
2026-04-03 22:21:18 Starting - Preparing the instances for training...
2026-04-03 22:21:41 Downloading - Downloading input data...
2026-04-03 22:22:22 Training - Training image download completed. Training in progress.Iniciando entrenamiento SageMaker
  n_estimators=400, learning_rate=0.05, max_depth=9
Datos cargados: 1,592,543 registros
Métricas | RMSE: 1.9704 | MAE: 0.8370 | R2: 0.7185
Catálogo copiado: items_catalog.parquet
Catálogo copiado: items_categories_catalog.parquet
Catálogo copiado: shops_catalog.parquet
Entrenamiento completo en 339.0s

2026-04-03 22:28:23 Uploading - Uploading generated training model
2026-04-03 22:28:36 Completed - Training job completed
Training seconds: 415
Billable seconds: 415
Training job completado


## 5. Deploy del endpoint de inferencia en tiempo real

SageMaker descarga el artefacto del modelo (`.tar.gz`) de S3, lo descomprime en `/opt/ml/model/`
e invoca el container con el comando `serve`, que arranca nginx → gunicorn → Flask.

In [9]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge",
    serializer=CSVSerializer()
)

print(f"Endpoint activo: {predictor.endpoint_name}")

INFO:sagemaker:Creating model with name: demand-retail-training-2026-04-03-22-29-23-234
INFO:sagemaker:Creating endpoint-config with name demand-retail-training-2026-04-03-22-29-23-234
INFO:sagemaker:Creating endpoint with name demand-retail-training-2026-04-03-22-29-23-234


----!Endpoint activo: demand-retail-training-2026-04-03-22-29-23-234


## 6. Predicciones en tiempo real

Enviamos un CSV al endpoint con las columnas `[ID, shop_id, item_id]`.
El predictor devuelve un CSV con `[ID, item_cnt_month]`.

In [10]:
# Carga el archivo de test
test_df = pd.read_csv(BASE_DIR / "data/raw/test.csv")
print(f"Registros de test: {len(test_df):,}")
test_df.head()

Registros de test: 214,200


,ID,shop_id,item_id
0,0,5,5037
1,1,5,5320
2,2,5,5233
3,3,5,5232
4,4,5,5268


In [13]:
# Tomamos una muestra pequeña para probar el endpoint
sample = test_df.head(50)

import io

# Enviar CSV con headers para que predictor.py encuentre las columnas por nombre
csv_buffer = io.StringIO()
sample.to_csv(csv_buffer, index=False)
csv_str = csv_buffer.getvalue()

response = predictor.predict(csv_str)

# Parsear el response
predictions = pd.read_csv(io.StringIO(response.decode("utf-8")))
print(f"Predicciones recibidas: {len(predictions)}")
predictions.head(10)

Predicciones recibidas: 50


,ID,item_cnt_month
0,0,1.821242
1,1,1.288914
2,2,1.571369
3,3,1.350493
4,4,1.358611
5,5,1.431305
6,6,1.719410
7,7,1.227286
8,8,1.574076
9,9,1.668559


In [15]:
# Predicciones sobre todo el test set
# Enviamos en lotes para evitar timeouts
BATCH_SIZE = 500
all_predictions = []

for i in range(0, len(test_df), BATCH_SIZE):
    batch = test_df.iloc[i:i + BATCH_SIZE]
    
    buf = io.StringIO()
    batch.to_csv(buf, index=False)
    
    response = predictor.predict(buf.getvalue())
    batch_pred = pd.read_csv(io.StringIO(response.decode("utf-8")))
    all_predictions.append(batch_pred)
    
    if i % 5000 == 0:
        print(f"Procesados {i}/{len(test_df)} registros...")

final_predictions = pd.concat(all_predictions, ignore_index=True)
final_predictions.to_csv(BASE_DIR /"data/predictions/predictions_realtime.csv", index=False)
print(f"Listo: {len(final_predictions):,} predicciones")

Procesados 0/214200 registros...
Procesados 5000/214200 registros...
Procesados 10000/214200 registros...
Procesados 15000/214200 registros...
Procesados 20000/214200 registros...
Procesados 25000/214200 registros...
Procesados 30000/214200 registros...
Procesados 35000/214200 registros...
Procesados 40000/214200 registros...
Procesados 45000/214200 registros...
Procesados 50000/214200 registros...
Procesados 55000/214200 registros...
Procesados 60000/214200 registros...
Procesados 65000/214200 registros...
Procesados 70000/214200 registros...
Procesados 75000/214200 registros...
Procesados 80000/214200 registros...
Procesados 85000/214200 registros...
Procesados 90000/214200 registros...
Procesados 95000/214200 registros...
Procesados 100000/214200 registros...
Procesados 105000/214200 registros...
Procesados 110000/214200 registros...
Procesados 115000/214200 registros...
Procesados 120000/214200 registros...
Procesados 125000/214200 registros...
Procesados 130000/214200 registros...

## 7. Limpieza — eliminar el endpoint

Los endpoints generan costo mientras están activos. Elimínalo cuando termines.

In [16]:
predictor.delete_endpoint()
print(f"Endpoint {predictor.endpoint_name} eliminado")

INFO:sagemaker:Deleting endpoint configuration with name: demand-retail-training-2026-04-03-22-29-23-234
INFO:sagemaker:Deleting endpoint with name: demand-retail-training-2026-04-03-22-29-23-234


Endpoint demand-retail-training-2026-04-03-22-29-23-234 eliminado
